# Continued Pretraining for Domain Adaptation

Continued pretraining, often called CPT or domain-adaptive pretraining, keeps training a language model on raw domain text before task-specific supervised fine-tuning. It is useful when the model needs more exposure to domain language, terminology, writing style, or code patterns.

This notebook focuses on the mechanics and decision-making: raw text datasets, causal language modeling, sequence packing, evaluation with held-out loss, and when CPT helps or hurts.

## What You Will Learn

By the end, you should understand:

- how continued pretraining differs from SFT,
- what raw text data should look like,
- how causal language modeling labels are created,
- why sequence packing matters,
- how to measure held-out domain loss,
- how CPT can cause forgetting,
- and how CPT fits before LoRA, QLoRA, or instruction tuning.

## 1. CPT vs SFT

| Method | Data | Objective | Learns |
|---|---|---|---|
| Continued pretraining | Raw domain text | Predict next token everywhere | Domain language distribution |
| Supervised fine-tuning | Prompt-response pairs | Predict target response tokens | Desired behavior format |
| Preference tuning | Chosen/rejected responses | Prefer chosen over rejected | Human preference style |

CPT is not primarily for teaching instruction following. It is for making the model more familiar with a domain. After CPT, you usually still do SFT or preference tuning.

In [ ]:
import importlib.util
import math
import random
from collections import Counter
from pathlib import Path

import pandas as pd

random.seed(42)

def has_package(name):
    return importlib.util.find_spec(name) is not None

print(f"transformers installed: {has_package('transformers')}")
print(f"datasets installed: {has_package('datasets')}")

## 2. Tiny Raw Domain Corpus

Raw domain text does not need prompt-response structure. It can be documentation, support tickets, policies, source code, clinical notes, legal clauses, logs, or product text. The key is that it should represent the language distribution you want the model to absorb.

In [ ]:
raw_documents = [
    "QLoRA fine-tuning loads the base language model in four-bit precision while training low-rank adapter weights.",
    "A fine-tuning run card should record base model, dataset version, tokenizer, maximum sequence length, and evaluation results.",
    "Adapter merging folds the learned low-rank update into the base model weights for simpler deployment.",
    "Domain-adaptive pretraining can improve terminology familiarity, but excessive training may cause catastrophic forgetting.",
    "Held-out loss on domain text helps detect whether continued pretraining is actually improving domain modeling.",
]

pd.DataFrame({"text": raw_documents, "words": [len(doc.split()) for doc in raw_documents]})

## 3. Data Quality Checks for Raw Text

CPT data has different risks from SFT data. There are no target answers to inspect, so you need checks for duplicate text, boilerplate, low-quality extraction, private data, and unwanted style.

In [ ]:
def raw_text_quality_report(documents):
    rows = []
    seen = Counter(documents)
    for index, text in enumerate(documents):
        rows.append({
            "index": index,
            "chars": len(text),
            "words": len(text.split()),
            "duplicate_count": seen[text],
            "looks_empty": not text.strip(),
            "has_email_like_text": "@" in text,
        })
    return pd.DataFrame(rows)

raw_text_quality_report(raw_documents)

## 4. A Simple Tokenizer for Concept Learning

Real CPT uses the base model tokenizer. To make packing visible, we will use a tiny whitespace tokenizer. The mechanics are the same: turn text into IDs, concatenate, split into fixed-length blocks, and train next-token prediction.

In [ ]:
class SimpleTokenizer:
    def __init__(self):
        self.token_to_id = {"<pad>": 0, "<eos>": 1, "<unk>": 2}
        self.id_to_token = {0: "<pad>", 1: "<eos>", 2: "<unk>"}

    def fit(self, texts):
        for text in texts:
            for token in text.lower().replace(".", " .").split():
                if token not in self.token_to_id:
                    index = len(self.token_to_id)
                    self.token_to_id[token] = index
                    self.id_to_token[index] = token

    def encode(self, text, add_eos=True):
        ids = [self.token_to_id.get(token, 2) for token in text.lower().replace(".", " .").split()]
        if add_eos:
            ids.append(self.token_to_id["<eos>"])
        return ids

    def decode(self, ids):
        return " ".join(self.id_to_token.get(index, "<unk>") for index in ids)

tokenizer = SimpleTokenizer()
tokenizer.fit(raw_documents)
encoded_docs = [tokenizer.encode(doc) for doc in raw_documents]
print(f"vocab_size: {len(tokenizer.token_to_id)}")
print(encoded_docs[0])
print(tokenizer.decode(encoded_docs[0]))

## 5. Causal LM Labels

For decoder-only language modeling, labels are usually the same sequence as inputs. The model internally shifts predictions so token position `t` predicts token `t + 1`. Unlike SFT, there is no assistant-only mask; raw text tokens all contribute unless they are padding.

In [ ]:
def make_causal_lm_example(token_ids, block_size):
    token_ids = token_ids[:block_size]
    pad_amount = block_size - len(token_ids)
    input_ids = token_ids + [0] * pad_amount
    attention_mask = [1] * len(token_ids) + [0] * pad_amount
    labels = token_ids + [-100] * pad_amount
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

example = make_causal_lm_example(encoded_docs[0], block_size=24)
pd.DataFrame(example).head(24)

## 6. Sequence Packing

If every document is padded separately, you waste compute on padding. Sequence packing concatenates tokenized documents and chunks them into fixed-length blocks. This improves token efficiency, especially when documents are short.

In [ ]:
def pack_documents(encoded_documents, block_size):
    flat = []
    for ids in encoded_documents:
        flat.extend(ids)
    usable_length = (len(flat) // block_size) * block_size
    blocks = [flat[i:i + block_size] for i in range(0, usable_length, block_size)]
    dropped = len(flat) - usable_length
    return blocks, dropped

packed_blocks, dropped_tokens = pack_documents(encoded_docs, block_size=16)
print(f"num_blocks: {len(packed_blocks)}")
print(f"dropped_tokens: {dropped_tokens}")
for block in packed_blocks[:2]:
    print(tokenizer.decode(block))

In [ ]:
def padding_waste(encoded_documents, block_size):
    padded_tokens = len(encoded_documents) * block_size
    real_tokens = sum(min(len(ids), block_size) for ids in encoded_documents)
    return 1 - real_tokens / padded_tokens

def packing_efficiency(encoded_documents, block_size):
    blocks, dropped = pack_documents(encoded_documents, block_size)
    real_tokens = len(blocks) * block_size
    total_tokens = sum(len(ids) for ids in encoded_documents)
    return real_tokens / total_tokens, dropped

rows = []
for block_size in [8, 16, 32]:
    efficiency, dropped = packing_efficiency(encoded_docs, block_size)
    rows.append({
        "block_size": block_size,
        "separate_padding_waste": padding_waste(encoded_docs, block_size),
        "packed_token_efficiency": efficiency,
        "dropped_tokens": dropped,
    })
pd.DataFrame(rows)

## 7. Train and Eval Splits for CPT

CPT should use held-out raw text. If held-out domain loss improves, the model is becoming better at predicting domain text. Still, lower domain loss does not guarantee better instruction behavior, so SFT and task evals remain necessary.

In [ ]:
def split_documents(documents, eval_fraction=0.2, seed=42):
    documents = list(documents)
    rng = random.Random(seed)
    rng.shuffle(documents)
    eval_size = max(1, round(len(documents) * eval_fraction))
    return documents[eval_size:], documents[:eval_size]

train_docs, eval_docs = split_documents(raw_documents)
print(f"train_docs: {len(train_docs)}")
print(f"eval_docs: {len(eval_docs)}")
print(eval_docs[0])

## 8. Hugging Face CPT Skeleton

This is the shape of a real continued pretraining run. It is guarded because real CPT requires `transformers`, `datasets`, enough data, and usually GPU compute.

In [ ]:
model_name = "sshleifer/tiny-gpt2"

try:
    from datasets import Dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments

    hf_tokenizer = AutoTokenizer.from_pretrained(model_name)
    if hf_tokenizer.pad_token is None:
        hf_tokenizer.pad_token = hf_tokenizer.eos_token

    dataset = Dataset.from_dict({"text": train_docs + eval_docs}).train_test_split(test_size=len(eval_docs), seed=42)

    def tokenize_batch(batch):
        return hf_tokenizer(batch["text"], truncation=True, max_length=128)

    tokenized = dataset.map(tokenize_batch, batched=True, remove_columns=["text"])
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.config.pad_token_id = hf_tokenizer.pad_token_id
    collator = DataCollatorForLanguageModeling(tokenizer=hf_tokenizer, mlm=False)

    args = TrainingArguments(
        output_dir="finetuning_outputs/continued_pretraining",
        per_device_train_batch_size=1,
        num_train_epochs=1,
        learning_rate=5e-5,
        eval_strategy="epoch",
        save_strategy="no",
        report_to=[],
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["test"],
        data_collator=collator,
    )
    print("CPT trainer is ready. Uncomment trainer.train() only when you want to run the demo.")
    # trainer.train()
except Exception as error:
    print("Hugging Face CPT skeleton skipped. Install transformers and datasets to run it.")
    print(f"Reason: {error}")

## 9. CPT Risks

| Risk | What happens | Mitigation |
|---|---|---|
| Catastrophic forgetting | Model loses general abilities | Mix in general data, train fewer steps, evaluate base behaviors |
| Domain overfitting | Model repeats domain text style too strongly | Deduplicate, hold out eval, stop early |
| Bad extraction quality | Model learns broken text artifacts | Clean and inspect raw text |
| Private data memorization | Model may leak sensitive snippets | filter PII, deduplicate, avoid secrets |
| No behavior improvement | Lower loss does not improve task answers | Follow with SFT and task eval |
| Compute waste | CPT is expensive for small gains | run small pilot first |

## 10. Decision Rule

Use CPT when:

- the base model struggles with domain terms, notation, or style,
- you have enough high-quality raw domain text,
- RAG alone does not solve the language mismatch,
- and you can evaluate both domain loss and downstream behavior.

Skip CPT when the model already understands the domain language and only needs to follow a task format. In that case, go straight to SFT or LoRA/QLoRA.

## Summary

Continued pretraining adapts a model to raw domain text using the same next-token objective as pretraining. The important skills are raw text quality checks, tokenization, sequence packing, held-out loss measurement, and deciding whether CPT is worth the risk of forgetting.

Next, we compare full fine-tuning, partial fine-tuning, linear probes, and adapter methods.